# Diagnóstico de calidad de los datos de Instacart

**Actividad:** GA1-220501094-01-AA1-EV06 – Diagnosticar la calidad de datos: preparación, exploración y documentación
**Rol:** analista de datos responsable de evaluar la calidad inicial de los datasets del proyecto (desarrollo individual)
**Aprendiz:** Juan Camilo Sarrazola Agudelo

> **Cómo está construido este notebook:** igual que en la EV04, las cifras de las interpretaciones no están escritas a mano.
> Se calculan con código a partir de los DataFrames cargados, de modo que el diagnóstico siempre corresponde a los datos.
> En esta actividad **no se limpia ni se modifica ningún registro**: solo se diagnostica, se interpreta y se documenta.

---
## Momento 1. Reconocimiento del proyecto y de los datasets

### 1.1 Nombre y propósito del proyecto
**Proyecto:** *Explorando Instacart: descubre patrones de compra.*

**Propósito:** describir los hábitos de compra de los clientes de Instacart (plataforma de compras de supermercado) a partir de
sus pedidos: cuándo compran, cada cuánto vuelven a comprar y qué productos repiten, para que el equipo de operaciones
pueda tomar decisiones con información confiable.

### 1.2 Pregunta analítica (recuperada de la EV04)
> **¿Qué hábitos de compra presentan los clientes de Instacart —cuándo compran, cada cuánto vuelven a comprar y qué productos recompran—?**

Se concreta en las tres preguntas de análisis formuladas en la EV04:
1. ¿Cada cuántos días, en promedio, vuelven a comprar los clientes? (`days_since_prior_order`)
2. ¿Qué proporción de los artículos de los pedidos son recompras? (`reordered`)
3. ¿En qué días y horas se concentran los pedidos de los clientes recurrentes? (`order_dow`, `order_hour_of_day`, `order_number`)

### 1.3 Datasets del proyecto y su relación con el problema

| N.º | Archivo | DataFrame | Qué contiene | Relación con la pregunta analítica |
|---|---|---|---|---|
| 1 | `instacart_orders.csv` | `orders` | Un registro por pedido: usuario, número de pedido, día, hora y días desde el pedido anterior. | Responde *cuándo* compran y *cada cuánto* vuelven. |
| 2 | `products.csv` | `products` | Catálogo: identificador, nombre, pasillo y departamento del producto. | Permite saber *qué* productos se compran. |
| 3 | `aisles.csv` | `aisles` | Tabla de consulta `aisle_id` → nombre del pasillo. | Agrupa productos por pasillo. |
| 4 | `departments.csv` | `departments` | Tabla de consulta `department_id` → nombre del departamento. | Agrupa productos por departamento. |
| 5 | `order_products.csv` | `order_products` | Detalle de pedidos: un registro por producto, posición en el carrito y si es recompra. | Responde *qué* se recompra y une pedidos con productos. |

### 1.4 Aspectos de calidad que se deben revisar antes de usar los datos
- **Completitud:** columnas con valores ausentes y si la ausencia tiene una explicación lógica.
- **Tipos de datos:** que cada variable tenga un tipo coherente con lo que representa (enteros, códigos, texto, binarios).
- **Unicidad:** filas completamente duplicadas y claves (`order_id`, `product_id`) repetidas.
- **Validez de rangos:** horas entre 0 y 23, días de la semana entre 0 y 6, `reordered` solo 0 o 1, etc.
- **Consistencia lógica:** que las variables no se contradigan entre sí (por ejemplo, primer pedido sin pedido anterior).
- **Integridad referencial:** que las claves de una tabla existan en la tabla con la que se relacionan.
- **Limitaciones:** información que falta para responder la pregunta analítica.

---
## Momento 2. Preparación del entorno y carga de datos

### 2.1 Librerías y funciones auxiliares

In [ ]:
import sys
import warnings
from pathlib import Path

import pandas as pd
from IPython.display import display, Markdown

pd.set_option("display.max_columns", None)
pd.set_option("display.max_colwidth", 90)


def md(texto):
    # Muestra un texto en formato Markdown dentro del notebook.
    display(Markdown(texto))


def fmt(n):
    # Entero con punto de miles (estilo es-CO): 478967 -> 478.967
    return f"{n:,.0f}".replace(",", ".")


def fmt2(x):
    # Decimal con coma decimal y punto de miles: 11.1017 -> 11,10
    return f"{x:,.2f}".replace(",", "X").replace(".", ",").replace("X", ".")


def pct(parte, total):
    # Porcentaje con dos decimales en formato es-CO.
    return fmt2(100 * parte / total)


print("Python:", sys.version.split()[0])
print("pandas:", pd.__version__)

### 2.2 Ubicación de los archivos
Los CSV no se versionan en GitHub (ver `README.md`). El notebook los busca primero en `data/raw/` (estructura de referencia de la guía)
y luego en `data/` (estructura usada desde la EV04). Si trabaja en Google Colab, agregue su ruta de Drive a `CANDIDATAS`.

In [ ]:
ARCHIVOS = {
    "orders": "instacart_orders.csv",
    "products": "products.csv",
    "aisles": "aisles.csv",
    "departments": "departments.csv",
    "order_products": "order_products.csv",
}

CANDIDATAS = [Path("../data/raw"), Path("../data"), Path("data/raw"), Path("data")]


def encontrar_ruta_datos():
    # Devuelve la primera carpeta que contiene los cinco CSV del proyecto.
    for carpeta in CANDIDATAS:
        if all((carpeta / archivo).exists() for archivo in ARCHIVOS.values()):
            return carpeta
    raise FileNotFoundError(
        "No se encontraron los cinco CSV de Instacart. Cópielos en data/raw/ o data/ (ver README.md)."
    )


RUTA_DATOS = encontrar_ruta_datos()
print("Carpeta de datos:", RUTA_DATOS.as_posix())

### 2.3 Carga de cada dataset en un DataFrame independiente
Los cinco archivos son texto delimitado que usa **punto y coma (`;`)** como separador, por eso se cargan con `pd.read_csv(..., sep=";")`.

In [ ]:
orders = pd.read_csv(RUTA_DATOS / ARCHIVOS["orders"], sep=";")
products = pd.read_csv(RUTA_DATOS / ARCHIVOS["products"], sep=";")
aisles = pd.read_csv(RUTA_DATOS / ARCHIVOS["aisles"], sep=";")
departments = pd.read_csv(RUTA_DATOS / ARCHIVOS["departments"], sep=";")
order_products = pd.read_csv(RUTA_DATOS / ARCHIVOS["order_products"], sep=";")

# Diccionario para recorrer los cinco DataFrames con un solo bucle
dataframes = {
    "orders": orders,
    "products": products,
    "aisles": aisles,
    "departments": departments,
    "order_products": order_products,
}

### 2.4 Verificación de la carga
Se comprueba que cada archivo produjo un DataFrame con más de una columna (si el separador fuera incorrecto, todo quedaría en **una sola columna**)
y con al menos un registro.

In [ ]:
verificacion = pd.DataFrame(
    [
        {
            "DataFrame": nombre,
            "archivo": ARCHIVOS[nombre],
            "tamaño archivo (MB)": round((RUTA_DATOS / ARCHIVOS[nombre]).stat().st_size / 1e6, 2),
            "filas": df.shape[0],
            "columnas": df.shape[1],
            "memoria en pandas (MB)": round(df.memory_usage(deep=True).sum() / 1e6, 1),
            "carga correcta": df.shape[0] > 0 and df.shape[1] > 1,
        }
        for nombre, df in dataframes.items()
    ]
)
assert verificacion["carga correcta"].all(), "Algún archivo no se cargó correctamente: revise el separador."
display(verificacion)
md(f"Los **{len(dataframes)}** datasets se cargaron correctamente y suman **{fmt(verificacion['filas'].sum())}** registros.")